# CatBoost Recommender for User-Specific History (CRUSH)

Self-made approach to the Catboost ranker using user history in retail

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from tecd_retail_recsys.data import download_tecd_data
from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.utils import calculate_avg_prices, calculate_overall_avg_price, get_avg_recs_price, get_item_to_price
from tecd_retail_recsys.utils import display_metrics_comparison
from tecd_retail_recsys.metrics import calculate_metrics
from tecd_retail_recsys.models import TopPopular, TopPersonal, EASE, iALS, TIFUKNN

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.


### Загрузка данных

In [3]:
dp = DataPreprocessor(day_begin=1082, day_end=1308, val_days=20, test_days=20, min_user_interactions=1, min_item_interactions=20)
train_df, val_df, test_df = dp.preprocess()

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 267,043,972 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (267043972, 12)
Filtered to 3,852,145 events with action_type='added_to_cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,327,057 events, 84,830 users, 32,095 items
Created mappings: 84830 users, 32095 items
Temporal split - Train: days < 1269 (922,368 events), Val: days 1269-1288 (232,900 events), Test: days >= 1289 (227,959 events)
Users in each part (train, val, test) - 7422


In [4]:
joined = dp.get_grouped_data(train_df, val_df, test_df)
joined['train_val_interactions'] = joined['train_interactions'] + joined['val_interactions']
print(joined.shape)

(7422, 5)


## Подготовка модели

### Разделение на трейн и валидацию

In [5]:
TRAIN_DF, VAL_DF = train_df[train_df['day'] < 1268 - 19], train_df[(train_df['day'] >= 1268 - 19)] 
JOINED = dp.get_grouped_data(TRAIN_DF, VAL_DF, VAL_DF)

### Обучение TopPersonal

In [6]:
toppers_val = TopPersonal()
toppers_val.fit(JOINED, col='train_interactions')
JOINED['toppers_recs'] = toppers_val.predict(JOINED, topn=100)

In [7]:
# добавлю фолбэк на TopPopular

toppop_val = TopPopular()
toppop_val.fit(JOINED, col='train_interactions')
JOINED['toppopular_recs_val'] = toppop_val.predict(JOINED, topn=100, return_scores=False)

def fill_to_k(df, primary_col, backup_col, target_len=100, out_col=None):
    out_col = out_col or primary_col

    def merge_lists(row):
        a = list(row[primary_col]) if row[primary_col] is not None else []
        b = list(row[backup_col]) if row[backup_col] is not None else []

        if len(a) >= target_len:
            return a[:target_len]

        need = target_len - len(a)
        return a + b[:need]

    df[out_col] = df.apply(merge_lists, axis=1)
    return df

JOINED = fill_to_k(
    JOINED,
    primary_col='toppers_recs',
    backup_col='toppopular_recs_val',
    target_len=100,
    out_col='toppers_recs_filled'
)

### Добавление таргетов

In [8]:
features_df = JOINED[['user_id', 'toppers_recs_filled']].explode('toppers_recs_filled')
features_df.rename(columns={'toppers_recs_filled': 'item_id'}, inplace=True)
VAL_DF['target'] = 1
features_df = features_df.merge(VAL_DF[['user_id', 'item_id', 'target']].drop_duplicates(), how='left').fillna(0)

In [9]:
features_df['target'].value_counts(True)

target
0.0    0.928569
1.0    0.071431
Name: proportion, dtype: float64

### Добавление фичей

In [10]:
from tecd_retail_recsys.data.features_builder import CandidateFeatureBuilder

In [11]:
builder = CandidateFeatureBuilder(TRAIN_DF)
features_df = builder.fit_transform(
    features_df,
    feature_groups=("user", "user_item", "item")
)

In [12]:
for col in ['item_brand_id', 'item_category', 'item_subcategory']:
    features_df[col] = features_df[col].cat.add_categories(['unknown']).fillna('unknown')

In [13]:
categorical_features = [
    "user_id",
    "item_id",
    "item_brand_id",
    "item_category",
    "item_subcategory"
]

features_df['user_id'] = features_df['user_id'].astype(np.int64)
features_df['item_id'] = features_df['item_id'].astype(np.int64)
features_df['item_brand_id'] = features_df['item_brand_id'].astype(str)

### Обучение кэтбуста

In [14]:
from catboost import CatBoostRanker, CatBoostClassifier

In [22]:
from catboost import CatBoostRanker, Pool

X, y = features_df.drop(columns=['target']), features_df['target']

train_pool = Pool(X, y, cat_features=categorical_features, group_id=X['user_id'])

model = CatBoostRanker(
    loss_function='YetiRank',
    eval_metric='NDCG:top=100',
    iterations=300,
    verbose=20,
    random_seed=42,
    
)

model.fit(
    train_pool,
    eval_set=train_pool,
    use_best_model=True,
    early_stopping_rounds=50
)


Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.5589144	best: 0.5589144 (0)	total: 229ms	remaining: 1m 8s
20:	test: 0.6465409	best: 0.6465409 (20)	total: 5.02s	remaining: 1m 6s
40:	test: 0.6529457	best: 0.6529663 (39)	total: 10.2s	remaining: 1m 4s
60:	test: 0.8291388	best: 0.8291388 (60)	total: 15.3s	remaining: 59.9s
80:	test: 0.9776116	best: 0.9780414 (79)	total: 20.5s	remaining: 55.4s
100:	test: 0.9995205	best: 0.9995205 (100)	total: 25.5s	remaining: 50.2s
120:	test: 0.9999511	best: 0.9999512 (119)	total: 30.5s	remaining: 45.2s
140:	test: 0.9999833	best: 0.9999833 (140)	total: 35.1s	remaining: 39.6s
160:	test: 1.0000000	best: 1.0000000 (145)	total: 39.9s	remaining: 34.4s
180:	test: 1.0000000	best: 1.0000000 (145)	total: 44.5s	remaining: 29.3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1
bestIteration = 145

Shrink model to first 146 iterations.


In [23]:
scores = model.predict(X)
X['score'] = scores

In [24]:
pd.DataFrame({'features': X.drop(columns=['score']).columns, 'imp': model.get_feature_importance(type='PredictionValuesChange')}).sort_values(by=['imp'], ascending=[False]).head(20)

,features,imp
47,user_item_share,18.745530
0,user_id,17.464410
23,user_item_last_ts,9.845241
24,user_item_active_span_days,9.757171
1,item_id,7.558099
39,item_events_per_active_day,6.400781
21,user_item_last_day,5.891616
40,item_repeat_user_ratio,4.851453
41,item_brand_id,4.747365
48,item_popularity_per_user,3.567860


In [25]:
val_recs = X.sort_values(by=['user_id', 'score'], ascending=[True, False])[['user_id', 'item_id']].groupby(['user_id'], as_index=False).agg(list)
val_recs.rename(columns={'item_id': 'crush_val_recs'}, inplace=True)
crush_val_metrics = calculate_metrics(joined.merge(val_recs), model_preds="crush_val_recs", gt_col="val_interactions", train_col="train_interactions", k_values=[100], verbose=True)

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (169501, 3) ratings_pred shape: (481400, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=30 gt_count=9 pred_count=100 overlap=2
  user_id=39 gt_count=8 pred_count=96 overlap=2
  user_id=51 gt_count=24 pred_count=99 overlap=6

At k=100:
  MAP@100       = 0.1705
  NDCG@100      = 0.4478
  Precision@100 = 0.0844
  Recall@100    = 0.2220

Other Metrics:
  MRR                 = 0.4716
  Catalog Coverage    = 0.9586
  Diversity     = 0.9967  [0=same recs for all, 1=unique recs]
  Novelty             = 0.6603
  Serendipity         = 0.0161


## Обучение модели на трейне+валидации

In [26]:
toppers_test = TopPersonal()
toppers_test.fit(joined, col='train_val_interactions')
joined['toppers_recs'] = toppers_test.predict(joined, topn=100)

In [27]:
# фолбэк

toppop_test = TopPopular()
toppop_test.fit(joined, col='train_val_interactions')
joined['toppopular_recs'] = toppop_test.predict(joined, topn=100, return_scores=False)

joined = fill_to_k(
    joined,
    primary_col='toppers_recs',
    backup_col='toppopular_recs',
    target_len=100,
    out_col='toppers_recs_filled'
)

In [28]:
features_df = joined[['user_id', 'toppers_recs_filled']].explode('toppers_recs_filled')
features_df.rename(columns={'toppers_recs_filled': 'item_id'}, inplace=True)

In [29]:
builder = CandidateFeatureBuilder(pd.concat([train_df, val_df]))
features_df = builder.fit_transform(
    features_df,
    feature_groups=("user", "user_item", "item")
)

In [30]:
for col in ['item_brand_id', 'item_category', 'item_subcategory']:
    features_df[col] = features_df[col].cat.add_categories(['unknown']).fillna('unknown')

In [31]:
categorical_features = [
    "user_id",
    "item_id",
    "item_brand_id",
    "item_category",
    "item_subcategory"
]

features_df['user_id'] = features_df['user_id'].astype(np.int64)
features_df['item_id'] = features_df['item_id'].astype(np.int64)
features_df['item_brand_id'] = features_df['item_brand_id'].astype(str)

In [32]:
scores = model.predict(features_df)
features_df['score'] = scores

test_recs = features_df.sort_values(by=['user_id', 'score'], ascending=[True, False])[['user_id', 'item_id']].groupby(['user_id'], as_index=False).agg(list)
test_recs.rename(columns={'item_id': 'crush_test_recs'}, inplace=True)

In [33]:
crush_test_metrics = calculate_metrics(joined.merge(test_recs), model_preds="crush_test_recs", gt_col="test_interactions", train_col="train_val_interactions", k_values=[100], verbose=True)

[Metrics debug] resolved gt_col='test_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (227959, 3) ratings_pred shape: (742200, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=12 gt_count=10 pred_count=94 overlap=3
  user_id=15 gt_count=54 pred_count=97 overlap=2
  user_id=23 gt_count=38 pred_count=98 overlap=12

At k=100:
  MAP@100       = 0.1534
  NDCG@100      = 0.4116
  Precision@100 = 0.0851
  Recall@100    = 0.2531

Other Metrics:
  MRR                 = 0.4094
  Catalog Coverage    = 0.9954
  Diversity     = 0.9969  [0=same recs for all, 1=unique recs]
  Novelty             = 0.6567
  Serendipity         = 0.0145


In [34]:
item_to_price = get_item_to_price(dp)
joined = joined.merge(test_recs)
avg_recs_price = get_avg_recs_price(joined, item_to_price, 'crush_test_recs')
print(f'Средняя цена рекомендаций модели CRUSH: {avg_recs_price:.2f}')

Средняя цена рекомендаций модели CRUSH: -3.98
